# Phase 2B.1 — retrieval union and blocker audit

Attach four Kaggle datasets: the raw seven TSV files, Phase 1 normalized artifacts, Phase 2A exact/structured artifacts, and Phase 2B multikey artifacts. Enable Internet, use a CPU session, then **Run All**. This notebook does not rerun retrieval or write a giant union table.

In [ ]:
from pathlib import Path
import json, subprocess, sys, time
REPO_URL='https://github.com/yugtheguy/NEW_AMAZON.git'
PINNED_COMMIT='714a87408c03a6c88479a54a9ecc2eaed7b852fa'
INPUT_ROOT=Path('/kaggle/input')
REPO_DIR=Path('/kaggle/working/NEW_AMAZON')
if not REPO_DIR.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(REPO_DIR)],check=True)
else: subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',PINNED_COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
assert actual==PINNED_COMMIT,(actual,PINNED_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[data]','duckdb==1.5.5'],check=True)
print('Active commit:',actual)

In [ ]:
SEARCH_ROOTS=[INPUT_ROOT,REPO_DIR/'artifacts']
def find_complete_root(stage,expected_count,label):
    counts={}
    for search_root in SEARCH_ROOTS:
        if not search_root.exists(): continue
        for manifest in search_root.rglob('part-*.manifest.json'):
            try: payload=json.loads(manifest.read_text())
            except Exception: continue
            if payload.get('stage')!=stage or len(manifest.parents)<4: continue
            root=manifest.parents[3].resolve()
            counts[root]=counts.get(root,0)+1
    complete=[root for root,count in counts.items() if count==expected_count]
    if len(complete)!=1:
        detail={str(root):count for root,count in counts.items()}
        raise RuntimeError(f'{label}: expected one complete root with {expected_count} manifests; found {detail}. Attach the dataset including both Parquet and .manifest.json files.')
    return complete[0]
NORMALIZED_ROOT=find_complete_root('normalization',89,'Phase 1')
EXACT_ROOT=find_complete_root('exact_structured',74,'Phase 2A')
BLOCKER_ROOT=find_complete_root('multikey_blocker',74,'Phase 2B')
expected_raw={'train_source1.tsv','train_source2.tsv','train_source3.tsv','train_ground_truth.tsv','test_source1.tsv','test_source2.tsv','test_source3.tsv'}
raw_matches={name:list(INPUT_ROOT.rglob(name)) for name in sorted(expected_raw)}
problems={name:paths for name,paths in raw_matches.items() if len(paths)!=1}
if problems: raise RuntimeError(f'Attach exactly one raw dataset; problems: {problems}')
print({'normalized':str(NORMALIZED_ROOT),'phase2a':str(EXACT_ROOT),'phase2b':str(BLOCKER_ROOT)})

In [ ]:
command=[sys.executable,'-m','amazon_er.cli','run','--stage','audit-phase2ab-union','--config',str(REPO_DIR/'configs/prod.yaml'),'--data-root',str(INPUT_ROOT),'--normalized-root',str(NORMALIZED_ROOT),'--exact-structured-root',str(EXACT_ROOT),'--multikey-root',str(BLOCKER_ROOT)]
log_path=Path('/kaggle/working/phase2b1_union_audit.log')
print('RUNNING:',' '.join(command),flush=True); started=time.monotonic()
with log_path.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=REPO_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
    code=process.wait()
if code: raise RuntimeError(f'Audit failed with exit code {code}; inspect {log_path}')
print(f'AUDIT COMPLETE in {time.monotonic()-started:.1f}s')

In [ ]:
REPORT_ROOT=REPO_DIR/'artifacts/reports/retrieval/v1/phase2ab_union_audit'
summary=json.loads((REPORT_ROOT/'summary.json').read_text())
print((REPORT_ROOT/'summary.md').read_text())
assert summary['status']=='complete'
print()
print('Reports:',REPORT_ROOT)
print('Use Save Version now to preserve summary.json, summary.md, and the log.')